# Get Raw Emissions Data

## Purpose

This notebook downloads and organizes the raw greenhouse gas emissions datasets required by the Geospatial-CANOE preprocessing workflow.

No data cleaning, spatial processing, facility aggregation, or CANOE schema encoding is performed here. The goal is only to acquire the raw emissions datasets and place them in the expected project directory structure.

---

## Data Sources

This workflow requires two complementary datasets describing Canada's large greenhouse gas emitting facilities.

### 1. Greenhouse Gas Emissions from Large Facilities (CSV)

- Source: Environment and Climate Change Canada
- Format: CSV
- Coverage: Annual greenhouse gas emissions reported by large industrial facilities across Canada.
- Purpose: Facility emissions attributes used during preprocessing.

Direct download:

```text
https://indicators-map.canada.ca/CSVs/en/Greenhouse%20gas%20emissions%20from%20large%20facilities%20-%202024.csv
```

---

### 2. Large Facility Geographic Locations (GeoJSON)

- Source: Environment and Climate Change Canada
- Format: GeoJSON distributed within a ZIP archive
- Coverage: Geographic locations of reporting facilities.
- Purpose: Spatial coordinates used to map facilities to CANOE regions.

The downloaded archive extracts to a folder containing:

```text
AirEmissions_GHG_2024.json
```

---

## Expected Output Structure

```text
data_files/
└── raw/
    └── emissions/
        └── co2_large_facilities_2024/
            ├── Greenhouse gas emissions from large facilities - 2024.csv
            └── AirEmissions_GHG_2024.json
```

---

## Workflow

This notebook performs the following stages:

1. Create the required raw emissions folder.
2. Define the CSV and GeoJSON download sources.
3. Download both datasets.
4. Extract the GeoJSON archive.
5. Flatten the extracted directory if necessary.
6. Validate that the expected CSV and GeoJSON files exist.

Later preprocessing notebooks assume these raw emissions datasets already exist.

In [1]:
# =============================================================================
# Cell 2 — Imports and project paths
# =============================================================================

from pathlib import Path
import shutil
import zipfile
import time

import requests


# Project root
PROJECT_ROOT = Path.cwd().parents[0]

# If running from a different working directory:
# PROJECT_ROOT = Path(r"C:\Users\aviga\Research\repos\temoa_geospace")


# Raw emissions directory
DATA_FILES = PROJECT_ROOT / "data_files"
RAW_DATA = DATA_FILES / "raw"
RAW_EMISSIONS = RAW_DATA / "emissions" / "co2_large_facilities_2024"


# Create required directory
RAW_EMISSIONS.mkdir(
    parents=True,
    exist_ok=True,
)


print(f"Project root: {PROJECT_ROOT}")
print(f"Raw emissions folder: {RAW_EMISSIONS}")

Project root: c:\Users\aviga\Research\repos\temoa_geospace
Raw emissions folder: c:\Users\aviga\Research\repos\temoa_geospace\data_files\raw\emissions\co2_large_facilities_2024


In [2]:
# =============================================================================
# Cell 3 — Download settings
# =============================================================================

HEADERS = {
    "User-Agent": (
        "Geospatial-CANOE/0.1 "
        "(University of Toronto Academic Research)"
    )
}

REQUEST_TIMEOUT = 120          # seconds
MAX_RETRIES = 3
DOWNLOAD_DELAY = 2             # seconds between downloads
CHUNK_SIZE = 1024 * 1024       # 1 MB streaming chunks

print("Download settings configured.")

Download settings configured.


In [3]:
# =============================================================================
# Cell 4 — Emissions source definition
# =============================================================================
# Environment and Climate Change Canada
#
# Source page:
# https://open.canada.ca/data/en/dataset/756bc907-34bb-4b33-9a87-b3c1a6c3f292
#
# This workflow requires two resources:
#   1. Facility emissions (CSV)
#   2. Facility locations (GeoJSON within ZIP archive)
#
# This cell only defines the download sources. It does not download anything.

EMISSIONS_SOURCE_PAGE = (
    "https://open.canada.ca/data/en/dataset/"
    "756bc907-34bb-4b33-9a87-b3c1a6c3f292"
)

EMISSIONS_RESOURCE = {
    "csv": {
        "name": "Large facility greenhouse gas emissions (CSV)",
        "url": (
            "https://indicators-map.canada.ca/CSVs/en/"
            "Greenhouse%20gas%20emissions%20from%20large%20facilities%20-%202024.csv"
        ),
        "filename": "Greenhouse gas emissions from large facilities - 2024.csv",
    },

    "geojson": {
        "name": "Large facility locations (GeoJSON)",
        "url": "https://indicators-map.canada.ca/historic/english/AirEmissions_GHG_2024.zip",
        "archive_name": "AirEmissions_GHG_2024.zip",
        "expected_file": "AirEmissions_GHG_2024.json",
    },
}

print("Emissions sources configured")
print(f"Source page: {EMISSIONS_SOURCE_PAGE}")

for key, resource in EMISSIONS_RESOURCE.items():
    print(f"\n{key.upper()}")
    print(f"  Name: {resource['name']}")
    print(f"  URL : {resource['url']}")

Emissions sources configured
Source page: https://open.canada.ca/data/en/dataset/756bc907-34bb-4b33-9a87-b3c1a6c3f292

CSV
  Name: Large facility greenhouse gas emissions (CSV)
  URL : https://indicators-map.canada.ca/CSVs/en/Greenhouse%20gas%20emissions%20from%20large%20facilities%20-%202024.csv

GEOJSON
  Name: Large facility locations (GeoJSON)
  URL : https://indicators-map.canada.ca/historic/english/AirEmissions_GHG_2024.zip


In [4]:
# =============================================================================
# Cell 5 — Download and extraction helper functions
# =============================================================================

def download_file(url, destination):
    """
    Download a file from a URL if it does not already exist.
    """

    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists():
        print(f"[Skip download] {destination.name} already exists.")
        return destination

    print(f"[Download] {destination.name}")

    response = requests.get(
        url,
        headers=HEADERS,
        stream=True,
        timeout=REQUEST_TIMEOUT,
    )

    response.raise_for_status()

    with open(destination, "wb") as f:
        for chunk in response.iter_content(CHUNK_SIZE):
            if chunk:
                f.write(chunk)

    print(f"[Complete download] {destination.name}")

    time.sleep(DOWNLOAD_DELAY)

    return destination


def extract_geojson_archive(zip_path, output_dir, overwrite=False):
    """
    Extract the GeoJSON archive into the raw emissions folder.
    Any nested folders are flattened.
    """

    output_dir.mkdir(parents=True, exist_ok=True)

    existing_json = sorted(output_dir.glob("*.json"))

    if existing_json and not overwrite:
        print(f"[Skip extract] {output_dir.name} already contains GeoJSON file(s).")
        return existing_json

    print(f"[Extract] {zip_path.name} → {output_dir.name}")

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(output_dir)

    json_files = sorted(output_dir.rglob("*.json"))

    if not json_files:
        raise FileNotFoundError(
            f"No GeoJSON file found after extracting {zip_path.name}"
        )

    for source_path in json_files:

        destination_path = output_dir / source_path.name

        if source_path.parent == output_dir:
            continue

        if destination_path.exists():
            if overwrite:
                destination_path.unlink()
            else:
                print(f"[Skip move] {destination_path.name} already exists.")
                continue

        shutil.move(str(source_path), str(destination_path))
        print(f"[Move] {source_path.name} → {destination_path.name}")

    for path in sorted(output_dir.iterdir()):
        if path.is_dir():
            shutil.rmtree(path)

    final_json = sorted(output_dir.glob("*.json"))

    if len(final_json) != 1:
        raise ValueError(
            f"Expected exactly 1 GeoJSON file in {output_dir}, "
            f"found {len(final_json)}"
        )

    if zip_path.exists():
        zip_path.unlink()
        print(f"[Delete] {zip_path.name}")

    print(f"[Complete] {output_dir.name}: {final_json[0].name}")

    return final_json

In [5]:
# =============================================================================
# Cell 6 — Download and extract emissions datasets
# =============================================================================

csv_path = RAW_EMISSIONS / EMISSIONS_RESOURCE["csv"]["filename"]

downloaded_csv = download_file(
    url=EMISSIONS_RESOURCE["csv"]["url"],
    destination=csv_path,
)

geojson_archive_path = RAW_EMISSIONS / EMISSIONS_RESOURCE["geojson"]["archive_name"]

downloaded_geojson_archive = download_file(
    url=EMISSIONS_RESOURCE["geojson"]["url"],
    destination=geojson_archive_path,
)

geojson_files = extract_geojson_archive(
    zip_path=downloaded_geojson_archive,
    output_dir=RAW_EMISSIONS,
    overwrite=False,
)

print("\nEmissions acquisition summary")
print("-----------------------------")

for file in sorted(RAW_EMISSIONS.iterdir()):
    if file.is_file():
        print(f"- {file.name}")

print(f"\nCSV ready: {downloaded_csv.name}")
print(f"GeoJSON ready: {geojson_files[0].name}")

[Download] Greenhouse gas emissions from large facilities - 2024.csv
[Complete download] Greenhouse gas emissions from large facilities - 2024.csv
[Download] AirEmissions_GHG_2024.zip
[Complete download] AirEmissions_GHG_2024.zip
[Extract] AirEmissions_GHG_2024.zip → co2_large_facilities_2024
[Delete] AirEmissions_GHG_2024.zip
[Complete] co2_large_facilities_2024: AirEmissions_GHG_2024.json

Emissions acquisition summary
-----------------------------
- AirEmissions_GHG_2024.json
- Greenhouse gas emissions from large facilities - 2024.csv

CSV ready: Greenhouse gas emissions from large facilities - 2024.csv
GeoJSON ready: AirEmissions_GHG_2024.json
